# ⚽ Predicting the FIFA World Cup 2026

## 📖 Regularized Dixon-Coles EV Simulator

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# Set seed for reproducible distributions
np.random.seed(2026) 

# 1. Load the original DataCamp templates
group_fixtures = pd.read_csv('data/group_fixtures.csv')
knockout_slots = pd.read_csv('data/knockout_slots.csv')

# 2. # Load finalized tournament predictions
my_preds = pd.read_csv('submissions/final_ev_submission.csv')

In [3]:
# --- 3. Process Group Stage (Matches 1-72) ---
group_predictions = group_fixtures.copy()
g_preds = my_preds[my_preds['match_id'] <= 72].copy()

if 'match_winner' in g_preds.columns:
    g_preds = g_preds.rename(columns={'match_winner': 'winning_team'})

g_cols = ['match_id', 'predicted_home_goals', 'predicted_away_goals', 'winning_team']
group_predictions = group_predictions.merge(g_preds[g_cols], on='match_id', how='left')

group_predictions['corners'] = np.random.poisson(lam=8.6, size=len(group_predictions)).clip(5, 13)
group_predictions['yellow_cards'] = np.random.poisson(lam=3.5, size=len(group_predictions)).clip(1, 7)
group_predictions['red_cards'] = 0 

In [4]:
# --- 4. Process Knockout Stage (Matches 73-104) ---
knockout_predictions = knockout_slots.copy()
k_preds = my_preds[my_preds['match_id'] > 72].copy()

if 'predicted_home_team' in k_preds.columns:
    t_home, t_away = 'predicted_home_team', 'predicted_away_team'
elif 'home_team' in k_preds.columns:
    t_home, t_away = 'home_team', 'away_team'

k_cols = ['match_id', 'predicted_home_goals', 'predicted_away_goals', 'match_winner', 'penalties', t_home, t_away]

knockout_predictions = knockout_predictions.merge(k_preds[k_cols], on='match_id', how='left')

if t_home == 'home_team':
    knockout_predictions = knockout_predictions.rename(columns={
        'home_team': 'predicted_home_team', 
        'away_team': 'predicted_away_team'
    })

knockout_predictions['corners'] = np.random.poisson(lam=8.8, size=len(knockout_predictions)).clip(5, 14)
knockout_predictions['yellow_cards'] = np.random.poisson(lam=3.9, size=len(knockout_predictions)).clip(2, 8)
knockout_predictions['red_cards'] = 0

In [19]:
# --- 5. Schema Validation ---

# Row count checks
assert len(group_predictions) == 72
assert len(knockout_predictions) == 32

# Missing value checks
assert group_predictions.isna().sum().sum() == 0
assert knockout_predictions.isna().sum().sum() == 0

print("Group Stage NaNs:", group_predictions.isna().sum().sum())
print("Knockout Stage NaNs:", knockout_predictions.isna().sum().sum())

print("\nSample Group Corners:", group_predictions['corners'].head().tolist())
print("Sample Group Yellows:", group_predictions['yellow_cards'].head().tolist())

group_predictions.head()
knockout_predictions.tail(16)



Group Stage NaNs: 0
Knockout Stage NaNs: 0

Sample Group Corners: [10, 9, 6, 11, 5]
Sample Group Yellows: [4, 5, 3, 2, 6]


,match_id,round,multiplier,slot_home,slot_away,predicted_home_goals,predicted_away_goals,match_winner,penalties,predicted_home_team,predicted_away_team,corners,yellow_cards,red_cards
16,89,Round of 16,2,Winner Match 73,Winner Match 75,1,1,away,True,Mexico,Germany,13,2,0
17,90,Round of 16,2,Winner Match 74,Winner Match 77,1,1,home,True,Brazil,Ecuador,6,4,0
18,91,Round of 16,2,Winner Match 76,Winner Match 78,1,1,away,True,Netherlands,France,12,6,0
19,92,Round of 16,2,Winner Match 79,Winner Match 80,1,1,away,True,Mexico,England,11,6,0
20,93,Round of 16,2,Winner Match 83,Winner Match 84,1,2,away,False,Croatia,Spain,10,2,0
21,94,Round of 16,2,Winner Match 81,Winner Match 82,1,1,home,True,Belgium,USA,8,4,0
22,95,Round of 16,2,Winner Match 86,Winner Match 88,1,2,away,False,Egypt,Portugal,10,8,0
23,96,Round of 16,2,Winner Match 85,Winner Match 87,1,2,away,False,Switzerland,Argentina,7,6,0
24,97,Quarter-final,4,Winner Match 89,Winner Match 90,1,1,away,True,Germany,Brazil,10,4,0
25,98,Quarter-final,4,Winner Match 93,Winner Match 94,2,1,home,False,Spain,Belgium,8,3,0


In [ ]:
# --- 6. Saving Predictions ---
PROJECT_ROOT = Path.cwd()

group_out = PROJECT_ROOT / "group_stage_FIFA2026.csv"
knockout_out = PROJECT_ROOT / "knockout_stage_FIFA2026.csv"

group_predictions.to_csv(group_out, index=False)
knockout_predictions.to_csv(knockout_out, index=False)

print(f"Saved group stage submission to: {group_out}")
print(f"Saved knockout stage submission to: {knockout_out}")

Saved group stage submission to: d:\Mrityunjay\Projects\FIfa2026\group_stage_FIFA2026.csv
Saved knockout stage submission to: d:\Mrityunjay\Projects\FIfa2026\knockout_stage_FIFA2026.csv
